In [ ]:
import os
import glob
import rasterio
import numpy as np
import cv2
from datetime import datetime

def extract_date_from_filename(filename):
    """
    פונקציה שמקבלת שם קובץ כמו 'ndvi_2026-01-02.tiff'
    ומחלצת מתוכו את התאריך כאובייקט של פייתון (כדי שנוכל למיין).
    """
    basename = os.path.basename(filename)
    # מורידים את הקידומת והסיומת כדי להישאר רק עם התאריך
    date_str = basename.replace("ndvi_", "").replace(".tiff", "")
    return datetime.strptime(date_str, "%Y-%m-%d")

def load_farm_images(base_data_folder):
    """
    עוברת על כל 4 תיקיות החוות, מושכת רק קבצי NDVI, 
    וממירה אותם למטריצות של Numpy שמוכנות ל-ML.
    """
    print(f"🔍 סורק את התיקייה הראשית: {base_data_folder}")
    
    # מוצא את כל התיקיות (החוות) בתוך התיקייה הראשית
    farm_folders = [f.path for f in os.scandir(base_data_folder) if f.is_dir()]
    all_farms_data = {}

    for folder in farm_folders:
        farm_name = os.path.basename(folder)
        
        # מחפש קבצים שמתחילים ב-ndvi ומסתיימים ב-tiff (מתעלם מ-ndre וכו')
        search_pattern = os.path.join(folder, "ndvi_*.tiff")
        tiff_files = glob.glob(search_pattern)

        # מסנן החוצה קבצי .aux.xml או קבצי מערכת אחרים שיש בתמונה שלך
        valid_tiffs = [f for f in tiff_files if not f.endswith('.xml') and not f.endswith('.Identifier')]
        
        # השלב הכי חשוב: מיון כרונולוגי של התמונות לפי התאריך בשם שלהן!
        valid_tiffs.sort(key=extract_date_from_filename)

        farm_images = []
        farm_dates = []

        for tiff_path in valid_tiffs:
            try:
                # פותחים את תמונת הלוויין
                with rasterio.open(tiff_path) as src:
                    img = src.read(1)  # קוראים את הערוץ הראשון (שמכיל את ה-NDVI)
                    
                    # הרשת שלנו צריכה גודל אחיד, למשל 64x64. אנחנו מכווצים את התמונה.
                    img_resized = cv2.resize(img, (64, 64))
                    
                    # אם יש פיקסלים "שבורים" (NaN), נהפוך אותם ל-0 כדי לא לקרוס
                    img_resized = np.nan_to_num(img_resized, nan=0.0)
                    
                    # נוסיף מימד נוסף בסוף כדי ש-TensorFlow יבין שזה ערוץ יחיד (שחור לבן)
                    # הגודל הופך מ-(64,64) ל-(64,64,1)
                    img_final = np.expand_dims(img_resized, axis=-1)

                    farm_images.append(img_final)
                    farm_dates.append(extract_date_from_filename(tiff_path).strftime("%Y-%m-%d"))
            except Exception as e:
                print(f"שגיאה בקריאת הקובץ {tiff_path}: {e}")

        # שומרים את הנתונים המוכנים במילון לפי שם החווה
        all_farms_data[farm_name] = {
            "images": np.array(farm_images),
            "dates": farm_dates
        }
        
        print(f"✅ נטענו {len(farm_images)} תמונות ממוינות עבור: {farm_name}")

    return all_farms_data

# --- בדיקה קטנה כדי לוודא שזה עובד ---
if __name__ == "__main__":
    # תחליף את המחרוזת הזו בנתיב המלא לתיקייה הראשית שבה שמת את 4 תיקיות החוות!
    BASE_DIR = "./my_satellite_data_folder" 
    
    # הפעלת הפונקציה
    dataset = load_farm_images(BASE_DIR)
    
    # הצגת מה שקיבלנו מהחווה הראשונה לדוגמה
    if dataset:
        first_farm = list(dataset.keys())[0]
        print(f"\nדוגמה לחלקה ({first_farm}):")
        print(f"צורה של מערך התמונות בזיכרון: {dataset[first_farm]['images'].shape}")
        print(f"תאריכי התמונות (מהישן לחדש): {dataset[first_farm]['dates']}")

🔍 סורק את התיקייה הראשית: ./my_satellite_data_folder


FileNotFoundError: [Errno 2] No such file or directory: './my_satellite_data_folder'